# ScamSense — Notebook 07: FAISS RAG Pipeline

Builds a **Retrieval-Augmented Generation** pipeline grounded in the
**SPF 2025 Annual Scams & Cybercrime Brief** — 100% free, no paid APIs.

| Step | What happens |
|---|---|
| 1 | SPF advisory chunks embedded with `paraphrase-multilingual-MiniLM-L12-v2` |
| 2 | FAISS index built (cosine similarity) |
| 3 | Query message → retrieve top-k relevant SPF passages |
| 4 | Template-based grounded explanation using retrieved passages |
| 5 | Full pipeline: Notebook 06 agent output + RAG grounding → final response |

**Output per query**: label + risk tier + SPF-grounded explanation with source citations  
**No LLM API needed** — all generation is template-based from retrieved text

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "numpy<2",
    "sentence-transformers==3.0.1",
    "langdetect",
    "langgraph",
    "transformers",
    "shap",
    "torch",
])

import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.0 MB/s eta 0:00:00


## Cell 2 — Imports & paths

In [1]:
import json, re, pickle
from pathlib import Path
import numpy as np
import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer

BASE_DIR   = Path("/kaggle/working/ScamSense")
RAG_DIR    = BASE_DIR / "rag"
RAG_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK ✅")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-06-11 13:07:23.095414: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781183243.118132     190 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781183243.125575     190 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781183243.145277     190 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the sam

Imports OK ✅


## Cell 3 — SPF 2025 advisory corpus
Extracted directly from: SPF Annual Scams & Cybercrime Brief 2025 (25 Feb 2026).  
Each chunk is a self-contained advisory passage with metadata.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# SPF 2025 Advisory Corpus
# Source: Singapore Police Force, Annual Scams & Cybercrime Brief 2025
# Published: 25 February 2026
# URL: https://www.police.gov.sg/media-room/statistics
# Each entry: {id, scam_type, topic, text, source_page}
# ─────────────────────────────────────────────────────────────────────────────

SPF_CORPUS = [
    # ── OVERALL SITUATION ────────────────────────────────────────────────────
    {
        "id": "spf_001",
        "scam_type": "overview",
        "topic": "Overall scam situation 2025",
        "text": (
            "In 2025, scam and cybercrime cases decreased by 24.8% to 41,974 cases, "
            "from 55,810 cases in 2024. Scam cases fell by 27.6% to 37,308 cases. "
            "Total losses from scams fell by 17.9% to about $913.1 million. "
            "Despite the decrease, the situation remains very concerning and tackling "
            "scams remains a key priority for the Singapore Government."
        ),
        "source_page": 1
    },
    {
        "id": "spf_002",
        "scam_type": "overview",
        "topic": "Self-effected transfers and social engineering",
        "text": (
            "The percentage of total reported scams involving self-effected transfers "
            "was 81.8% in 2025. Scammers did not gain direct control of victims' accounts "
            "but manipulated them into performing monetary transactions through deception "
            "and social engineering. The SPF urges the public to exercise caution when "
            "making fund transfers, such as by verifying the legitimacy of such requests."
        ),
        "source_page": 4
    },
    {
        "id": "spf_003",
        "scam_type": "overview",
        "topic": "Cryptocurrency losses 2025",
        "text": (
            "Cryptocurrency losses accounted for about $182.2 million, or about 20% of "
            "total scam losses in 2025. Scammers leverage cryptocurrency because of its "
            "irreversible transactions and limited traceability, making asset recovery "
            "very challenging. Tether, Ethereum, and Bitcoin were the top cryptocurrencies "
            "lost, accounting for about 91.7% of total cryptocurrency scam losses."
        ),
        "source_page": 3
    },
    {
        "id": "spf_004",
        "scam_type": "overview",
        "topic": "Top scam types by cases 2025",
        "text": (
            "In 2025, the top five scam types by number of cases were: "
            "e-commerce scams (6,703 cases, 18.0%), phishing scams (6,264 cases, 16.8%), "
            "job scams (5,575 cases, 14.9%), investment scams (5,462 cases, 14.6%), "
            "and government officials impersonation scams (3,363 cases, 9.0%)."
        ),
        "source_page": 4
    },
    {
        "id": "spf_005",
        "scam_type": "overview",
        "topic": "Top scam types by losses 2025",
        "text": (
            "In 2025, the top five scam types by amount lost were: "
            "investment scams ($336.2M, 36.8%), government officials impersonation scams "
            "($242.9M, 26.6%), job scams ($123.5M, 13.5%), phishing scams ($39.9M, 4.4%), "
            "and business email compromise scams ($35.3M, 3.9%). "
            "Total scam losses in 2025 were $913.1 million."
        ),
        "source_page": 5
    },

    # ── INVESTMENT SCAMS ─────────────────────────────────────────────────────
    {
        "id": "spf_006",
        "scam_type": "investment",
        "topic": "Investment scam statistics 2025",
        "text": (
            "Investment scams recorded the highest losses among all scam types in 2025: "
            "$336.2 million, a 4.8% increase from $320.7 million in 2024. "
            "There were 5,462 investment scam cases in 2025, a decrease of 19.8% from "
            "6,814 cases in 2024. The average loss per victim was $61,559 — "
            "the highest average loss among the top scam types."
        ),
        "source_page": 7
    },
    {
        "id": "spf_007",
        "scam_type": "investment",
        "topic": "Investment scam tactics — Telegram and Facebook",
        "text": (
            "Investment scam victims came across opportunities through recommendations "
            "from online friends, internet searches, online advertisements, or were added "
            "into chat groups via messaging platforms. The most common platforms used by "
            "investment scammers were Telegram and Facebook. Victims were enticed by false "
            "testimonies and instructed to transfer money to specified bank accounts or "
            "make payments via bank cards for investing."
        ),
        "source_page": 18
    },
    {
        "id": "spf_008",
        "scam_type": "investment",
        "topic": "Investment scam — cryptocurrency tactics",
        "text": (
            "Investment scams accounted for 38.4% of all scam cases involving "
            "cryptocurrency transactions in 2025. Scammers directed victims to create "
            "new cryptocurrency accounts, fund them with money from bank accounts, "
            "buy cryptocurrency, then transfer it to scammer-controlled accounts. "
            "In some cases, victims shared their cryptocurrency account login credentials "
            "and seed phrase with scammers, allowing scammers to drain the account."
        ),
        "source_page": 18
    },
    {
        "id": "spf_009",
        "scam_type": "investment",
        "topic": "Investment scam — fake apps and withdrawal fees",
        "text": (
            "Scammers asked victims to download shell investment apps from official app "
            "stores. These apps promoted fraudulent investment products such as fake "
            "cryptocurrency, forex, and stock-trading products, and showed fictitious "
            "investment progress and returns. Victims discovered they were scammed only "
            "when they experienced difficulties withdrawing returns despite transferring "
            "increasingly large sums of fees or taxes."
        ),
        "source_page": 19
    },

    # ── GOVERNMENT IMPERSONATION ─────────────────────────────────────────────
    {
        "id": "spf_010",
        "scam_type": "impersonation",
        "topic": "Government impersonation scam statistics 2025",
        "text": (
            "Government officials impersonation scams more than doubled in 2025, "
            "increasing by 123.6% to 3,363 cases from 1,504 cases in 2024. "
            "It recorded the second highest losses among all scam types: $242.9 million, "
            "a 60.5% increase from $151.3 million in 2024. "
            "The average loss per victim was $72,229 — the highest of all scam types."
        ),
        "source_page": 6
    },
    {
        "id": "spf_011",
        "scam_type": "impersonation",
        "topic": "Government impersonation — bank and local official variant",
        "text": (
            "In 91.7% of government impersonation cases, victims received unsolicited "
            "calls from scammers posing as bank or financial institution representatives "
            "citing suspicious transactions. Victims were then transferred to fake "
            "government officials who accused them of money laundering and instructed "
            "them to transfer money to safety accounts or hand over cash for "
            "investigation purposes."
        ),
        "source_page": 17
    },
    {
        "id": "spf_012",
        "scam_type": "impersonation",
        "topic": "Singapore government officials will never ask these things",
        "text": (
            "Singapore Government officials will never ask members of the public to do "
            "the following over a phone call: ask you to transfer money; ask you to "
            "disclose banking login details; ask you to install mobile apps from "
            "unofficial app stores; or transfer your call to the Police. "
            "Never transfer monies, hand monies or valuables to any unknown person "
            "or person whose identity you did not verify."
        ),
        "source_page": 18
    },
    {
        "id": "spf_013",
        "scam_type": "impersonation",
        "topic": "Government impersonation — new PayNow and crypto transfer tactics",
        "text": (
            "New trends observed in 2025 for government impersonation scams include: "
            "transfer of funds from victims' bank accounts to Payment Service Provider "
            "accounts (e.g. YouTrip) controlled by scammers via PayNow; and "
            "cryptocurrency transfers by requesting victims to create a new "
            "cryptocurrency account, fund it from their bank account, and transfer "
            "cryptocurrency to scammer-controlled accounts."
        ),
        "source_page": 17
    },

    # ── JOB SCAMS ────────────────────────────────────────────────────────────
    {
        "id": "spf_014",
        "scam_type": "job",
        "topic": "Job scam statistics 2025",
        "text": (
            "Job scams recorded 5,575 cases in 2025, a significant decrease of 38.4% "
            "from 9,043 cases in 2024. Total losses were $123.5 million, down from "
            "$156.2 million in 2024. The average loss per victim was $22,163. "
            "Job scams were the third highest scam type by both number of cases and "
            "total losses in 2025."
        ),
        "source_page": 16
    },
    {
        "id": "spf_015",
        "scam_type": "job",
        "topic": "Job scam tactics — upfront fees and fake tasks",
        "text": (
            "Job scammers typically advertise easy work-from-home jobs with high pay "
            "on social media and messaging platforms. Victims are asked to pay "
            "registration fees, training fees, or deposits before starting work. "
            "Some scams involve completing online tasks such as liking posts or "
            "boosting products, where initial small payments are made to build trust "
            "before victims are asked for large sums that are never returned."
        ),
        "source_page": 16
    },
    {
        "id": "spf_016",
        "scam_type": "job",
        "topic": "Job scam — victim age profile",
        "text": (
            "Job scams disproportionately affected young adults aged 20 to 29, "
            "who made up 19.9% of all scam victims in 2025, with 17.8% of young adult "
            "victims falling prey to job scams. Adults aged 30 to 49 also had high "
            "exposure, with 17.6% falling prey to job scams. "
            "Legitimate employers do not ask for upfront fees before employment."
        ),
        "source_page": 9
    },

    # ── PHISHING SCAMS ───────────────────────────────────────────────────────
    {
        "id": "spf_017",
        "scam_type": "phishing",
        "topic": "Phishing scam statistics 2025",
        "text": (
            "Phishing scams recorded the second highest number of cases in 2025: "
            "6,264 cases, a 26.8% decrease from 8,552 cases in 2024. "
            "Total losses decreased by 32.8% to $39.9 million from $59.4 million in 2024. "
            "The average loss per phishing victim was $6,384."
        ),
        "source_page": 7
    },
    {
        "id": "spf_018",
        "scam_type": "phishing",
        "topic": "Phishing scam tactics — card details and OTP theft",
        "text": (
            "Phishing scams in 2025 predominantly involved unauthorised card transactions "
            "where victims unknowingly submitted their card details and authentication "
            "codes via OTP or digital token to scammers, to complete seemingly legitimate "
            "purchases. Victims were directed to fake websites impersonating banks, "
            "government agencies, or e-commerce platforms."
        ),
        "source_page": 7
    },
    {
        "id": "spf_019",
        "scam_type": "phishing",
        "topic": "Phishing contact methods — SMS and email links",
        "text": (
            "Phishing scammers predominantly contact victims via SMS, email, and "
            "messaging platforms containing links to fraudulent websites. "
            "The major retail banks have phased out SMS one-time passwords for "
            "authentication of debit or credit card transactions for digital token users. "
            "Never click on links in unsolicited messages — go directly to your bank's "
            "official website or app."
        ),
        "source_page": 11
    },

    # ── E-COMMERCE SCAMS ─────────────────────────────────────────────────────
    {
        "id": "spf_020",
        "scam_type": "ecommerce",
        "topic": "E-commerce scam statistics 2025",
        "text": (
            "E-commerce scams recorded the highest number of cases among all scam types "
            "in 2025: 6,703 cases, despite a 42.5% decrease from 11,665 cases in 2024. "
            "Total losses were $16.7 million, down 4.6% from $17.5 million in 2024. "
            "The average loss per victim was $2,503."
        ),
        "source_page": 6
    },
    {
        "id": "spf_021",
        "scam_type": "ecommerce",
        "topic": "E-commerce scam platforms and payment methods",
        "text": (
            "Carousell and Facebook Marketplace were the platforms most commonly used "
            "by e-commerce scammers, accounting for 29.0% and 22.2% of cases respectively. "
            "After expressing interest in a product, victims were instructed to make "
            "payment or an initial deposit via PayNow or bank transfer. They realised "
            "they had been scammed when they failed to receive the products or when "
            "sellers became uncontactable."
        ),
        "source_page": 6
    },
    {
        "id": "spf_022",
        "scam_type": "ecommerce",
        "topic": "E-commerce scam — Pokemon cards most common item",
        "text": (
            "Pokemon trading cards were the item most commonly involved in e-commerce "
            "scams in 2025, making up 13.6% of e-commerce cases. "
            "Other commonly scammed items included electronics, concert tickets, and "
            "luxury goods. Victims should use platforms' protected payment features "
            "and meet sellers in person for high-value transactions."
        ),
        "source_page": 6
    },

    # ── LOVE SCAMS ───────────────────────────────────────────────────────────
    {
        "id": "spf_023",
        "scam_type": "love",
        "topic": "Internet love scam statistics 2025",
        "text": (
            "Internet love scams recorded 917 cases in 2025, up 7.6% from 852 cases "
            "in 2024. Total losses were $24.9 million, down slightly from $27.6 million. "
            "The average loss per love scam victim was $27,202 — significantly higher "
            "than e-commerce or phishing scams, reflecting the extended trust-building "
            "process scammers employ."
        ),
        "source_page": 16
    },
    {
        "id": "spf_024",
        "scam_type": "love",
        "topic": "Love scam tactics — online relationship and money requests",
        "text": (
            "Love scammers build relationships with victims over weeks or months through "
            "dating apps, social media, or messaging platforms. They create convincing "
            "personas, often claiming to be overseas professionals. Once sufficient trust "
            "is established, they fabricate emergencies or investment opportunities "
            "requiring the victim to send money. Red flags include refusal to video call, "
            "claims of being stranded overseas, and requests for urgent fund transfers."
        ),
        "source_page": 16
    },

    # ── CONTACT METHODS ──────────────────────────────────────────────────────
    {
        "id": "spf_025",
        "scam_type": "overview",
        "topic": "Online platforms used by scammers 2025",
        "text": (
            "Online platforms were used by scammers to reach victims in 84.1% of all "
            "scam cases in 2025. Meta platforms were involved in 35.4% of all cases, "
            "with Facebook alone accounting for 18.0%. The top contact methods were "
            "social media (10,448 cases), messaging platforms (9,355 cases), "
            "phone calls (5,477 cases), and online shopping platforms (3,804 cases)."
        ),
        "source_page": 8
    },
    {
        "id": "spf_026",
        "scam_type": "overview",
        "topic": "WhatsApp and Telegram used most for scam contact",
        "text": (
            "Among messaging platforms, WhatsApp accounted for 53.5% of cases, "
            "Telegram for 37.9%, and Facebook Messenger for 4.2%. "
            "Among social media, Facebook accounted for 51.9%, TikTok 26.0%, "
            "and Instagram 14.2% of scam contact cases."
        ),
        "source_page": 8
    },

    # ── VICTIM PROFILE ───────────────────────────────────────────────────────
    {
        "id": "spf_027",
        "scam_type": "overview",
        "topic": "Scam victim age profile 2025",
        "text": (
            "In 2025, 85.2% of scam victims were aged below 65. "
            "Adults aged 30-49 made up 36.1% of victims, average loss $22,283. "
            "Young seniors aged 50-64 made up 23.6%, average loss $32,983. "
            "The elderly aged 65+ made up 14.8%, with the highest average loss "
            "of $37,053 per victim, predominantly falling prey to investment scams, "
            "government impersonation scams, and phishing scams."
        ),
        "source_page": 9
    },

    # ── ANTI-SCAM ACTIONS ─────────────────────────────────────────────────────
    {
        "id": "spf_028",
        "scam_type": "overview",
        "topic": "ScamShield helpline and reporting",
        "text": (
            "When in doubt, members of the public should call the ScamShield Helpline "
            "at 1799, or check the ScamShield application for verification. "
            "Report scam cases to the Singapore Police Force at 1800-255-0000 or "
            "online at police.gov.sg/i-witness. "
            "Report phishing links to report@scamalert.sg. "
            "Check investment legitimacy at the MAS Investor Alert List: mas.gov.sg."
        ),
        "source_page": 13
    },
    {
        "id": "spf_029",
        "scam_type": "overview",
        "topic": "Anti-scam measures — caning and Protection from Scams Act",
        "text": (
            "From 30 December 2025, caning for scam-related offences was operationalised. "
            "Scammers face mandatory caning of at least 6 strokes, up to 24 strokes. "
            "Scam mules who launder proceeds may face up to 12 strokes. "
            "The Protection from Scams Act, operationalised on 1 July 2025, allows the "
            "SPF to issue Restriction Orders to banks to restrict banking facilities of "
            "scam victims who continue to believe scammers despite police engagement."
        ),
        "source_page": 10
    },
    {
        "id": "spf_030",
        "scam_type": "overview",
        "topic": "Mule accounts — do not share bank accounts or Singpass",
        "text": (
            "Criminal syndicates perpetrate scams in Singapore through local money mules. "
            "The public must not allow anyone to use their financial accounts to transfer "
            "funds, provide access to Singpass credentials, or supply local SIM cards. "
            "In 2025, more than 7,000 money mules and scammers were investigated, "
            "and more than 940 have been charged. Individuals found guilty of mule-related "
            "crimes may face substantial prison sentences, monetary penalties, and caning."
        ),
        "source_page": 13
    },
]

print(f"SPF corpus loaded: {len(SPF_CORPUS)} advisory chunks")
print(f"Scam types covered: {set(c['scam_type'] for c in SPF_CORPUS)}")

SPF corpus loaded: 30 advisory chunks
Scam types covered: {'investment', 'impersonation', 'job', 'phishing', 'love', 'ecommerce', 'overview'}


## Cell 4 — Embed corpus and build FAISS index

In [4]:
# paraphrase-multilingual handles all 5 language varieties
embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Embed all corpus texts
corpus_texts = [c["text"] for c in SPF_CORPUS]
print("Embedding SPF corpus...")
corpus_embeddings = embedder.encode(
    corpus_texts,
    normalize_embeddings=True,   # L2 norm → cosine similarity via inner product
    show_progress_bar=True
).astype(np.float32)

DIM = corpus_embeddings.shape[1]
print(f"Embedding dim: {DIM}")

# Build FAISS index (inner product on L2-normalised = cosine similarity)
index = faiss.IndexFlatIP(DIM)
index.add(corpus_embeddings)
print(f"FAISS index built: {index.ntotal} vectors")

# Save index + corpus for downstream notebooks
faiss.write_index(index, str(RAG_DIR / "spf_faiss.index"))
with open(RAG_DIR / "spf_corpus.json", "w") as f:
    json.dump(SPF_CORPUS, f, indent=2)
print("✅ FAISS index saved → rag/spf_faiss.index")
print("✅ Corpus saved     → rag/spf_corpus.json")

Embedding SPF corpus...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding dim: 384
FAISS index built: 30 vectors
✅ FAISS index saved → rag/spf_faiss.index
✅ Corpus saved     → rag/spf_corpus.json


## Cell 5 — RAG retrieval function

In [6]:
def retrieve(query: str, top_k: int = 3, scam_type_filter: str = None) -> list[dict]:
    """
    Retrieve top-k SPF advisory chunks for a query.
    Optional scam_type_filter: only return chunks of that type.
    Returns list of {id, scam_type, topic, text, source_page, score}.
    """
    q_embed = embedder.encode(
        [query], normalize_embeddings=True
    ).astype(np.float32)

    if scam_type_filter:
        # Filter index to only chunks of the given scam type
        filtered_ids = [
            i for i, c in enumerate(SPF_CORPUS)
            if c["scam_type"] == scam_type_filter
        ]
        if not filtered_ids:
            filtered_ids = list(range(len(SPF_CORPUS)))
        filtered_embeds = corpus_embeddings[filtered_ids]
        scores = (q_embed @ filtered_embeds.T)[0]
        top_local = np.argsort(scores)[::-1][:top_k]
        results = []
        for local_i in top_local:
            global_i = filtered_ids[local_i]
            c = SPF_CORPUS[global_i].copy()
            c["score"] = round(float(scores[local_i]), 4)
            results.append(c)
    else:
        scores_arr, indices = index.search(q_embed, top_k)
        results = []
        for score, idx in zip(scores_arr[0], indices[0]):
            c = SPF_CORPUS[idx].copy()
            c["score"] = round(float(score), 4)
            results.append(c)

    return results


# Quick test
hits = retrieve("guaranteed returns crypto investment telegram", top_k=3)
print("Test retrieval for investment query:")
for h in hits:
    print(f"  [{h['score']:.3f}] {h['id']} — {h['topic']}")

Test retrieval for investment query:
  [0.506] spf_003 — Cryptocurrency losses 2025
  [0.409] spf_008 — Investment scam — cryptocurrency tactics
  [0.366] spf_002 — Self-effected transfers and social engineering


## Cell 6 — Grounded explanation generator (template-based, no LLM API)

In [9]:
def generate_grounded_explanation(
    message: str,
    label: str,
    confidence: float,
    language: str,
    scam_type: str,
    risk_tier: str,
    risk_score: int,
    top_features: list,
    top_k: int = 3,
) -> dict:
    """
    Generates a grounded explanation by:
    1. Retrieving top-k relevant SPF advisory chunks
    2. Building a structured explanation citing SPF sources
    Returns dict with retrieved chunks + formatted explanation.
    """
    if label == "ham":
        return {
            "explanation": "This message appears legitimate. No scam indicators detected.",
            "retrieved"  : [],
            "sources"    : []
        }

    # Retrieve SPF chunks — prefer matching scam type, fallback to general
    type_hits    = retrieve(message, top_k=top_k, scam_type_filter=scam_type)
    general_hits = retrieve(message, top_k=2, scam_type_filter=None)

    # Combine, deduplicate by id, keep top (top_k + 1)
    seen = set()
    all_hits = []
    for h in (type_hits + general_hits):
        if h["id"] not in seen:
            seen.add(h["id"])
            all_hits.append(h)
    all_hits = all_hits[: top_k + 1]

    # Build SHAP feature string
    if top_features:
        key_tokens = ", ".join(
            f"'{f['token']}'" for f in top_features[:3]
        )
        shap_line = f"Key suspicious tokens identified: {key_tokens}."
    else:
        shap_line = ""

    # Pull the most relevant SPF passage (highest score)
    best = all_hits[0] if all_hits else None
    stats_chunk = next(
        (h for h in all_hits if "statistics" in h["topic"].lower()), best
    )
    advice_chunk = next(
        (h for h in all_hits
         if any(w in h["topic"].lower()
                for w in ["never", "reporting", "helpline", "advice", "action"])
        ),
        None
    )

    # Compose explanation
    tier_phrases = {
        "CRITICAL": "⚠️ CRITICAL RISK",
        "HIGH"    : "🔴 HIGH RISK",
        "MEDIUM"  : "🟠 MEDIUM RISK",
        "LOW"     : "🟡 LOW RISK",
    }
    tier_phrase = tier_phrases.get(risk_tier, risk_tier)

    lines = [
        f"{tier_phrase} — ScamSense classified this message as a scam "
        f"({confidence:.1%} confidence, language: {language}, risk score: {risk_score}/100).",
        "",
    ]

    if shap_line:
        lines.append(shap_line)
        lines.append("")

    if stats_chunk:
        lines.append(
            f"According to the SPF 2025 Annual Scams Brief (p.{stats_chunk['source_page']}): "
            f"{stats_chunk['text']}"
        )
        lines.append("")

    if advice_chunk and advice_chunk["id"] != (stats_chunk["id"] if stats_chunk else None):
        lines.append(
            f"SPF advisory (p.{advice_chunk['source_page']}): "
            f"{advice_chunk['text']}"
        )
        lines.append("")

    lines.append(
        "What to do: Do NOT respond, click any links, or transfer money. "
        "Report to SPF at 1800-255-0000 or check ScamShield at 1799."
    )

    sources = [
        f"SPF 2025 Brief p.{h['source_page']} — {h['topic']}"
        for h in all_hits
    ]

    return {
        "explanation": "\n".join(lines),
        "retrieved"  : all_hits,
        "sources"    : sources,
    }

print("Grounded explanation function ready ✅")

Grounded explanation function ready ✅


## Cell 7 — Full RAG pipeline (Notebook 06 agents + RAG grounding)

In [19]:
# Re-import Notebook 06 components
# In production these would be imported from a shared module.
# Here we inline the necessary pieces for a self-contained notebook.

import re, torch
from pathlib import Path
from typing import TypedDict, Optional
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import shap
from langdetect import detect, LangDetectException
from langgraph.graph import StateGraph, END
from kaggle_secrets import UserSecretsClient
_secrets = UserSecretsClient()

HF_TOKEN = _secrets.get_secret("HF_TOKEN")
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

# Load classifier (reuse if already loaded, else reload)
try:
    clf_model
    print("Classifier already loaded")
except NameError:
    HF_MODEL_ID   = "Bhoovika/scamsense-xlmroberta"
    clf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, token=HF_TOKEN)
    clf_model     = AutoModelForSequenceClassification.from_pretrained(
        HF_MODEL_ID, token=HF_TOKEN
    ).to(DEVICE)
    clf_model.eval()
    print("Classifier loaded")

LABEL_MAP = {0: "ham", 1: "scam"}

def run_classifier(text):
    inputs = clf_tokenizer(text, return_tensors="pt", truncation=True,
                           max_length=128, padding=True).to(DEVICE)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred  = int(probs.argmax())
    return {"label": LABEL_MAP[pred], "confidence": round(float(probs[pred]), 4),
            "scam_prob": round(float(probs[1]), 4)}

SINGLISH_RE = re.compile(
    r"\blah\b|\bleh\b|\bsia\b|\blor\b|\bwah\b|\baiyo\b|\bdun\b|\bwan\b|\bone\b",
    re.IGNORECASE
)
def detect_language(text):
    hits = len(SINGLISH_RE.findall(text))
    if len(text.split()) > 0 and hits / len(text.split()) >= 0.08:
        return "singlish"
    try:
        d = detect(text)
        return {"en": "en", "ms": "ms", "id": "ms", "ta": "ta",
                "zh-cn": "zh", "zh-tw": "zh", "zh": "zh"}.get(d, "en")
    except:
        return "en"

SPF_TAX = {
    "investment":    {"risk_score": 90},
    "impersonation": {"risk_score": 88},
    "job":           {"risk_score": 75},
    "phishing":      {"risk_score": 70},
    "ecommerce":     {"risk_score": 60},
    "loan":          {"risk_score": 65},
    "romance":       {"risk_score": 72},
    "unknown":       {"risk_score": 50},
}

print("SPF_TAX loaded ✅", type(SPF_TAX))

# Load SPF taxonomy
with open(BASE_DIR / "rag"/ "spf_corpus.json") as f:
    SPF_CORPUS = json.load(f)

SPF_KW = {
    "investment" : [r"invest", r"return", r"profit", r"crypto", r"trading",
                    r"guaranteed", r"bitcoin", r"ethereum", r"wallet", r"roi"],
    "impersonation": [r"police", r"spf", r"officer", r"arrest", r"investigation",
                      r"money laundering", r"safety account", r"警察", r"公安"],
    "job"        : [r"job", r"work from home", r"part.?time", r"salary",
                    r"earn", r"recruit", r"registration fee", r"kerja"],
    "phishing"   : [r"click", r"verify", r"suspend", r"otp", r"login",
                    r"http", r"\.xyz", r"bank", r"password", r"singpass"],
    "ecommerce"  : [r"sell", r"cheap", r"carousell", r"shopee", r"paynow first",
                    r"ship", r"legit", r"transfer first"],
    "love"       : [r"love", r"relationship", r"dating", r"overseas",
                    r"send money", r"stranded", r"emergency"],
}

def get_scam_type(text):
    tl = text.lower()
    scores = {t: sum(1 for kw in kws if re.search(kw, tl))
              for t, kws in SPF_KW.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "unknown"

def get_risk(scam_type, scam_prob):
    base = SPF_TAX.get(scam_type, SPF_TAX["unknown"])["risk_score"]  # was SPF_CORPUS
    score = max(10, min(100, int(base * scam_prob)))
    tier  = ("CRITICAL" if score >= 85 else "HIGH" if score >= 65
             else "MEDIUM" if score >= 40 else "LOW")
    return score, tier

# SHAP
def shap_predict(texts):
    if isinstance(texts, str): texts = [texts]
    inputs = clf_tokenizer(list(texts), return_tensors="pt",
                           truncation=True, max_length=128, padding=True).to(DEVICE)
    with torch.no_grad():
        logits = clf_model(**inputs).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

masker    = shap.maskers.Text(clf_tokenizer)
explainer = shap.Explainer(shap_predict, masker, output_names=["ham", "scam"])

def get_top_shap(text, n=5):
    sv = explainer([text])
    pairs = [{"token": t, "shap_value": round(float(v), 4)}
             for t, v in zip(sv.data[0], sv.values[0, :, 1])
             if t not in ["", "▁", "<s>", "</s>", "<pad>"]]
    return sorted(pairs, key=lambda x: x["shap_value"], reverse=True)[:n]


def run_full_rag_pipeline(message: str, verbose: bool = True) -> dict:
    """Full pipeline: detect → classify → SHAP → SPF risk → RAG grounding."""
    # Step 1: detect + classify
    lang   = detect_language(message)
    clf    = run_classifier(message)
    label, conf, scam_prob = clf["label"], clf["confidence"], clf["scam_prob"]

    # Step 2: SHAP (only for scam)
    top_features = get_top_shap(message) if label == "scam" else []

    # Step 3: SPF risk scoring
    scam_type = get_scam_type(message) if label == "scam" else None
    risk_score, risk_tier = (get_risk(scam_type, scam_prob)
                             if label == "scam" else (0, "NONE"))

    # Step 4: RAG grounded explanation
    rag = generate_grounded_explanation(
        message=message, label=label, confidence=conf,
        language=lang, scam_type=scam_type or "unknown",
        risk_tier=risk_tier, risk_score=risk_score,
        top_features=top_features,
    )

    result = {
        "message"     : message,
        "language"    : lang,
        "label"       : label,
        "confidence"  : conf,
        "scam_type"   : scam_type,
        "risk_score"  : risk_score,
        "risk_tier"   : risk_tier,
        "top_features": top_features,
        "explanation" : rag["explanation"],
        "sources"     : rag["sources"],
        "retrieved"   : rag["retrieved"],
    }

    if verbose:
        icons = {"CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡", "LOW": "🟢", "NONE": "✅"}
        print(f"{icons.get(risk_tier, '❓')} [{risk_tier}] {label.upper()} "
              f"| conf={conf:.1%} | lang={lang} | type={scam_type}")
        print(f"\n{rag['explanation']}")
        if rag["sources"]:
            print("\n📚 Sources:")
            for s in rag["sources"]:
                print(f"   • {s}")
        print("-" * 72)

    return result

print("Full RAG pipeline ready ✅")

Classifier already loaded
SPF_TAX loaded ✅ <class 'dict'>
Full RAG pipeline ready ✅


## Cell 8 — Inference examples: one per language + scam type

In [20]:
test_cases = [
    ("en / phishing",
     "Your POSB account has been suspended. Verify details at http://posb-secure.xyz or it will be closed."),
    ("en / investment",
     "Guaranteed 300% crypto returns in 30 days. Join our Telegram investment group now."),
    ("ms / job scam",
     "Kerja mudah dari rumah! Pendapatan RM5,000 sebulan. Bayar yuran pendaftaran RM200 dahulu."),
    ("ta / investment",
     "உங்கள் முதலீட்டில் 300% லாபம் உறுதி. இப்போதே பதிவு செய்யுங்கள்!"),
    ("zh / impersonation",
     "您好，我是新加坡警察局侦探。您的账户涉嫌洗钱，请立即转账至安全账户。"),
    ("singlish / ecommerce",
     "Selling iPhone 15 Pro Max $400 only lah! PayNow me first then I ship lor. Legit one wan!"),
    ("en / ham",
     "Hi! Your dentist appointment is tomorrow 2pm at Raffles Dental. Reply to confirm."),
]

print("=" * 72)
print("SCAMSENSE RAG PIPELINE — GROUNDED INFERENCE RESULTS")
print("=" * 72)

all_results = []
for name, msg in test_cases:
    print(f"\n▶ {name}")
    print(f"  Input: {msg[:85]}...")
    print()
    r = run_full_rag_pipeline(msg, verbose=True)
    r["test_case"] = name
    all_results.append(r)

print("\n✅ All inference examples complete")

SCAMSENSE RAG PIPELINE — GROUNDED INFERENCE RESULTS

▶ en / phishing
  Input: Your POSB account has been suspended. Verify details at http://posb-secure.xyz or it ...

🟠 [HIGH] SCAM | conf=100.0% | lang=en | type=phishing

🔴 HIGH RISK — ScamSense classified this message as a scam (100.0% confidence, language: en, risk score: 70/100).

Key suspicious tokens identified: 'POS', 'closed', 'account '.

According to the SPF 2025 Annual Scams Brief (p.7): Phishing scams recorded the second highest number of cases in 2025: 6,264 cases, a 26.8% decrease from 8,552 cases in 2024. Total losses decreased by 32.8% to $39.9 million from $59.4 million in 2024. The average loss per phishing victim was $6,384.

What to do: Do NOT respond, click any links, or transfer money. Report to SPF at 1800-255-0000 or check ScamShield at 1799.

📚 Sources:
   • SPF 2025 Brief p.11 — Phishing contact methods — SMS and email links
   • SPF 2025 Brief p.7 — Phishing scam tactics — card details and OTP theft
   • SPF 

## Cell 9 — Summary table + save results

In [21]:
rows = []
for r in all_results:
    rows.append({
        "test_case"    : r["test_case"],
        "label"        : r["label"],
        "confidence"   : f"{r['confidence']:.1%}",
        "language"     : r["language"],
        "scam_type"    : r.get("scam_type") or "—",
        "risk_tier"    : r["risk_tier"],
        "risk_score"   : r["risk_score"],
        "sources_count": len(r.get("sources", [])),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

out_dir = BASE_DIR / "results"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "07_rag_inference_results.csv", index=False)

# Also save full results with explanations as JSON
save_results = []
for r in all_results:
    save_results.append({
        k: v for k, v in r.items() if k != "retrieved"
    })
with open(out_dir / "07_rag_inference_full.json", "w") as f:
    json.dump(save_results, f, indent=2, ensure_ascii=False)

print("\n✅ Results saved → results/07_rag_inference_results.csv")
print("✅ Full JSON    → results/07_rag_inference_full.json")

           test_case label confidence language     scam_type risk_tier  risk_score  sources_count
       en / phishing  scam     100.0%       en      phishing      HIGH          70              4
     en / investment  scam     100.0%       en    investment  CRITICAL          90              4
       ms / job scam  scam     100.0%       ms           job      HIGH          75              4
     ta / investment  scam     100.0%       ta       unknown    MEDIUM          50              3
  zh / impersonation  scam      77.9%       zh impersonation      HIGH          68              4
singlish / ecommerce  scam     100.0% singlish     ecommerce    MEDIUM          60              4
            en / ham   ham      98.5%       en             —      NONE           0              0

✅ Results saved → results/07_rag_inference_results.csv
✅ Full JSON    → results/07_rag_inference_full.json


## Cell 10 — Save to Drive + push to GitHub

In [22]:
import shutil, subprocess

# Copy RAG index to Drive
drive_rag = Path("/kaggle/working/")
drive_rag.mkdir(parents=True, exist_ok=True)
shutil.copy(RAG_DIR / "spf_faiss.index", drive_rag / "spf_faiss.index")
shutil.copy(RAG_DIR / "spf_corpus.json", drive_rag / "spf_corpus.json")
print("✅ FAISS index + corpus saved to Drive")


print("\n" + "=" * 60)
print("✅ Notebook 07 complete!")
print("   FAISS index : 30 SPF 2025 advisory chunks")
print("   Retrieval   : paraphrase-multilingual-MiniLM-L12-v2")
print("   Grounding   : template-based, SPF page citations")
print("   Cost        : $0 — no paid APIs")
print("   Next        : Notebook 08 — FastAPI backend")
print("=" * 60)

✅ FAISS index + corpus saved to Drive

✅ Notebook 07 complete!
   FAISS index : 30 SPF 2025 advisory chunks
   Retrieval   : paraphrase-multilingual-MiniLM-L12-v2
   Grounding   : template-based, SPF page citations
   Cost        : $0 — no paid APIs
   Next        : Notebook 08 — FastAPI backend
